# B-13: TFT and TabPFN for recursive rollout

Fills in the two remaining stretch items from B-09 (D-53) — TFT (deferred for known instability,
D-45/D-48) and TabPFN (deferred, unresearched) — now attempted against the same recursive-rollout
backtest as B-09-B12, Tower 4, single-anchor (2021-12-16) smoke test formalized here; 5-anchor
(2018-2022) sweeps run as script extensions (`b13a_tft_multi_anchor.py`,
`b13b_tabpfn_multi_anchor.py`), same precedent as every prior B-09-B12 notebook.

**Part A** (not in this notebook): extended the actual/gap-filled/predicted chain-plot visualization
(`results/figures/b10_chains/`) to DLinear/LSTM across all 3 towers x 5 years — pure visualization,
no new metrics, via `dl_chain_plots.py`.

**Part B — TFT**: reuses D-45's exact regularized recipe (`d_model=32, n_heads=4,
weight_decay=1e-3, patience=5`) — no new HPO. One adaptation: B-03b validated against a full
held-out calendar year; recursive-rollout anchors don't have that much data to spare (anchor=2018
has only 715 pre-anchor days total), so this reserves the **last 90 days before the anchor** as
the validation slice instead — a data-availability necessity, not a tuning choice.

**Part C — TabPFN**: `tabpfn-time-series` is **not autoregressive** — it predicts the entire
365-day horizon in a single forward pass from a context (history) + future (known covariates)
dataframe, architecturally closer to SARIMAX's one-shot `get_forecast` than to
`tree_rollout`/`dl_rollout`'s day-by-day loop. Runs in **local inference mode** (one-time browser
license acceptance at ux.priorlabs.ai, `TABPFN_TOKEN` env var, then all inference on the local
GPU — no per-call data transmission to Prior Labs, user-confirmed). Context uses real
`y_observed` (gaps as NaN, TabPFN handles missing values internally) rather than `y_gapfilled` —
deliberately avoiding the diffuse globally-trained-gap-filler optimism flagged for every other
model's training target (D-53's own caveat), since TabPFN's "context" isn't supervised training
in the same sense.

In [1]:
from pathlib import Path
import os, sys, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")

import models.forecasting_dl as fdl
import models.recursive_rollout as rr

HOURLY = Path("../../data/Hourly"); RESULTS = Path("../../results")
ANCHOR = pd.Timestamp("2021-12-16")
N_DAYS = 365
TOWER = 4
VAL_DAYS = 90
target_dates = pd.date_range(ANCHOR + pd.Timedelta(days=1), periods=N_DAYS, freq="D")

dv = pd.read_csv(HOURLY/"forecast_daily_v2.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
FX_B = [c for c in dv.columns if c.startswith("fx")]
df4 = dv[dv.tower == TOWER].set_index("Datetime").sort_index()
anchor_val = df4.loc[ANCHOR, "y_gapfilled"]
persist = rr.chain_persistence(anchor_val, N_DAYS)
print(f"Anchor: {ANCHOR.date()} -> target window {target_dates[0].date()} .. {target_dates[-1].date()}")

Anchor: 2021-12-16 -> target window 2021-12-17 .. 2022-12-16


## Part B — TFT: build, train (regularized), roll out

In [2]:
device = fdl.get_device(); print("device:", device)
track = "B"; cfg = fdl.TRACKS[track]
m = fdl.load_matrix(HOURLY/"forecast_features_v2.csv")

cutoff = ANCHOR + pd.Timedelta(hours=23, minutes=59)
val_start = ANCHOR - pd.Timedelta(days=VAL_DAYS)

W = fdl.build_windows(m, track)
train_parts, val_parts = [], []
for t in [2, 4, 9]:
    ttime = pd.DatetimeIndex(W[t]["ttime"][:, -1])
    train_parts.append(fdl._subset(W[t], ttime <= val_start))
    val_parts.append(fdl._subset(W[t], (ttime > val_start) & (ttime <= cutoff)))
train_tft = fdl._cat(train_parts); val_tft = fdl._cat(val_parts)
print(f"TFT train windows: {len(train_tft['enc'])}, val windows: {len(val_tft['enc'])} "
      f"(val = last {VAL_DAYS}d before anchor)")

se_t, sd_t = fdl.Scaler().fit(train_tft["enc"]), fdl.Scaler().fit(train_tft["dec"])
yv = train_tft["y"][np.isfinite(train_tft["y"])]
mu_t, sdy_t = float(yv.mean()), float(yv.std() + 1e-6)
train_tft["enc"], train_tft["dec"] = se_t.tf(train_tft["enc"]), sd_t.tf(train_tft["dec"])
val_tft["enc"], val_tft["dec"] = se_t.tf(val_tft["enc"]), sd_t.tf(val_tft["dec"])
n_enc, n_dec = train_tft["enc"].shape[-1], train_tft["dec"].shape[-1]

t0 = time.time()
tft_model = fdl.build_model("TFT", cfg["L"], cfg["H"], n_enc, n_dec, 3)
fdl.train_model(tft_model, train_tft, device, epochs=30, ch4_mu=mu_t, ch4_sd=sdy_t, seed=0,
                 weight_decay=1e-3, val_data=val_tft, patience=5)
print(f"TFT trained ({time.time()-t0:.0f}s)")

val_pred = fdl.predict(tft_model, val_tft, device, mu_t, sdy_t)
print(f"val prediction sanity: any_nan={np.isnan(val_pred).any()}, "
      f"mean={np.nanmean(val_pred):.1f} (real mean={mu_t:.1f}), "
      f"std={np.nanstd(val_pred):.1f} (real std={sdy_t:.1f})")

ser4 = fdl.tower_series(m, TOWER, track)
dates_full, enc_ex_full, dec_ex_full = ser4["idx"], ser4["enc_ex"], ser4["dec_ex"]
Y_full_real, ch4_full_real = ser4["Y"], ser4["ch4"]
anchor_idx = dates_full.get_loc(ANCHOR)
history_init_tft = ch4_full_real[:anchor_idx + 1]

tft_chain = rr.dl_rollout(tft_model, se_t, sd_t, mu_t, sdy_t, device, fdl.TOW[TOWER],
                           enc_ex_full, dec_ex_full, dates_full, history_init_tft, ANCHOR,
                           L=cfg["L"], H=cfg["H"], n_days=N_DAYS)

y_true_full = pd.Series(Y_full_real, index=dates_full)
y_true = y_true_full.reindex(target_dates).values
bm_tft = rr.bin_metrics(y_true, tft_chain.reindex(target_dates).values, target_dates, ANCHOR, y_persist=persist)
bm_tft["model"] = "TFT"
bm_tft.to_csv(RESULTS/"b13a_tft_summary.csv", index=False)
tft_chain.to_frame("TFT").to_csv(RESULTS/"b13a_tft_chain.csv")
print(bm_tft.to_string(index=False))

device: cuda


TFT train windows: 5040, val windows: 270 (val = last 90d before anchor)


TFT trained (4s)
val prediction sanity: any_nan=False, mean=22.9 (real mean=32.1), std=33.2 (real std=64.6)


    bin  n      R2    MAE   MASE model
    1-7  3 -11.391  8.486 3.4187   TFT
   8-30 19  -0.501 14.962 1.1772   TFT
  31-90 50  -0.094 13.272 1.0221   TFT
 91-180 90   0.275 52.283 0.8057   TFT
181-270 88   0.224 62.423 0.9205   TFT
271-365 91  -0.415 27.943 1.3414   TFT


## Part C — TabPFN: one-shot 365-day forecast

Requires `TABPFN_TOKEN` set in the environment (one-time browser license acceptance).

In [3]:
assert os.environ.get("TABPFN_TOKEN"), "TABPFN_TOKEN not set -- see https://ux.priorlabs.ai/account/licenses"

hist = df4.loc[:ANCHOR]
hist_target = hist["y_observed"]          # real only, gaps as NaN -- see markdown above
hist_covariates = hist[FX_B]
future_covariates = df4.loc[target_dates, FX_B]
print(f"context: {len(hist_target)} days ({hist_target.notna().sum()} real), "
      f"future: {len(future_covariates)} days, {len(FX_B)} covariates")

t0 = time.time()
tabpfn_chain = rr.tabpfn_forecast(hist_target, hist_covariates, future_covariates, mode="local")
print(f"TabPFN forecast done ({time.time()-t0:.0f}s)")

y_true2 = df4.reindex(target_dates)["y_observed"].values
bm_tabpfn = rr.bin_metrics(y_true2, tabpfn_chain.reindex(target_dates).values, target_dates, ANCHOR, y_persist=persist)
bm_tabpfn["model"] = "TabPFN"
bm_tabpfn.to_csv(RESULTS/"b13b_tabpfn_summary.csv", index=False)
tabpfn_chain.to_frame("TabPFN").to_csv(RESULTS/"b13b_tabpfn_chain.csv")
print(bm_tabpfn.to_string(index=False))

context: 1811 days (1010 real), future: 365 days, 34 covariates


GPU 0::   0%|          | 0/1 [00:00<?, ?it/s]

GPU 0:: 100%|██████████| 1/1 [00:01<00:00,  1.57s/it]

GPU 0:: 100%|██████████| 1/1 [00:01<00:00,  1.57s/it]

TabPFN forecast done (4s)
    bin  n     R2    MAE   MASE  model
    1-7  3 -0.733  3.027 1.2194 TabPFN
   8-30 19  0.105 11.531 0.9073 TabPFN
  31-90 50  0.006 12.437 0.9578 TabPFN
 91-180 90 -0.007 49.504 0.7629 TabPFN
181-270 88  0.061 61.872 0.9124 TabPFN
271-365 91  0.164 19.700 0.9457 TabPFN


## Multi-anchor (2018-2022) extension — the actual verdicts

Both single-anchor results above were stable/non-degenerate (no NaNs, no catastrophic bins), so
both were extended to the same 5-anchor sweep as script extensions (`b13a_tft_multi_anchor.py`,
`b13b_tabpfn_multi_anchor.py`, not re-executed here). Results:
`results/b13a_tft_multi_anchor.csv`, `results/b13b_tabpfn_multi_anchor.csv`. Full interpretation
in `b13_results.md`. Headline (n-weighted mean R2/MASE across 5 anchors, Tower 4):

- **TFT**: mean R2 = -0.237, mean MASE = 1.055 -- avoids the original unregularized-TFT
  catastrophic-failure pathology entirely (D-45's fix generalizes to this new recursive-rollout
  context), lands as the best DL model in the B-09-B13 sequence (beats LSTM -0.438, DLinear
  -1.460), but does not beat the tree/SARIMAX models.
- **TabPFN**: mean R2 = -0.006, mean MASE = **0.862** -- the best mean MASE of *any* model tested
  across the entire B-09-B13 sequence (beats B-10's ensemble at 0.975, XGB at 0.968), and a mean
  R2 competitive with LightGBM/SARIMAX/RF, achieved with **zero training/HPO** (in-context
  learning only). A genuinely notable finding: a zero-shot foundation model is essentially
  competitive with a carefully-tuned tree ensemble on this task.